# Data Loading 

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv"

# Read the CSV file and parse the Date column as datetime objects
df = pd.read_csv(url, parse_dates=['Date'])
temps = df['Temp'].values
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

temps = scaler.fit_transform(
    temps.reshape(-1, 1)
).flatten()
import torch

def create_sequences(data, seq_length):
    X, y = [], []

    for i in range(len(data) - seq_length):
        # Input sequence
        X.append(data[i:i + seq_length])

        # Target value (next temperature)
        y.append(data[i + seq_length])

    return torch.tensor(X, dtype=torch.float32), \
           torch.tensor(y, dtype=torch.float32)
    seq_length = 10

X, y = create_sequences(temps, seq_length)
X = X.unsqueeze(-1)

# Model 1: RNN for Time Series Forecasting

In [ ]:
import torch
import torch.nn as nn
# Define an RNN-based prediction model by inheriting from nn.Module
class RNNPredictor(nn.Module):
    def __init__(self, input_size=1, hidden_size=32):
        super().__init__()  # Call the constructor of the parent class
        # Define an RNN layer
        # input_size: dimensionality of input features
        # hidden_size: number of hidden units in the RNN
        # batch_first=True means the input tensor shape is:
        # [batch_size, sequence_length, input_size]
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        # Define a fully connected layer to map the RNN hidden state
        # to the prediction output (a single scalar value)
        self.fc = nn.Linear(hidden_size, 1)
    # Define the forward propagation process
    def forward(self, x):
        # Input shape:
        # [batch_size, sequence_length, input_size]
        # h_n is the hidden state of the last time step
        # Shape: [1, batch_size, hidden_size]
        _, h_n = self.rnn(x)
        # Remove the first dimension
        # From [1, batch_size, hidden_size]
        # To   [batch_size, hidden_size]
        #
        # Then pass it through the fully connected layer to obtain
        # the final output with shape [batch_size, 1]
        return self.fc(h_n.squeeze(0))
# Instantiate the model using the default parameters:
# input_size = 1, hidden_size = 32
model = RNNPredictor()
# Define the loss function
# Mean Squared Error (MSE) is commonly used for regression tasks
loss_fn = nn.MSELoss()
# Define the optimizer
# Adam optimization algorithm with a learning rate of 0.01
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Model Training

In [ ]:
# Train the model for 50 epochs
for epoch in range(50):
    # Set the model to training mode
    # (enabling training-specific mechanisms such as dropout)
    model.train()
    # Forward pass:
    # Feed the input data X into the model to obtain predictions
    pred = model(X)  # Output shape: [batch_size, 1]
    # Compute the Mean Squared Error (MSE) loss
    # between the predicted values and the ground-truth labels
    loss = loss_fn(pred.squeeze(), y)
    # pred.squeeze() removes the extra dimension so that
    # its shape matches y ([batch_size])
    # Clear previously accumulated gradients
    # to prevent gradient accumulation across iterations
    optimizer.zero_grad()
    # Backward pass:
    # Compute gradients of the loss with respect to model parameters
    loss.backward()
    # Update model parameters using the optimizer
    optimizer.step()
    # Print the loss value every 10 epochs
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


# Model Evaluation (Prediction Visualization)

In [ ]:
import matplotlib.pyplot as plt  # Import the plotting library for result visualization
# Set the model to evaluation mode
# (disabling training-specific behaviors such as dropout)
model.eval()
# Generate predictions using the trained model
# - detach(): Detach the tensor from the computation graph to prevent gradient tracking
# - squeeze(): Remove unnecessary dimensions (e.g., [batch_size, 1] -> [batch_size])
# - numpy(): Convert the tensor to a NumPy array for visualization
preds = model(X).detach().squeeze().numpy()
# Convert the ground-truth labels to a NumPy array
true = y.numpy()
# Plot only the first 100 samples for comparison
plt.plot(preds[:100], label='Predicted')  # Predicted values
plt.plot(true[:100], label='True')        # Ground-truth values
# Add a legend and title
plt.legend()
plt.title("RNN Temperature Prediction")
# Display the figure
plt.show()

# Model 2: Transformer for Character-Level Text Generation

In [ ]:
import requests
import torch
import torch.nn as nn

# ============================================================
# Read the Tiny Shakespeare dataset
# ============================================================
text = requests.get(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
).text

# Build the vocabulary
chars = sorted(list(set(text)))

# Character-to-index mapping
stoi = {ch: i for i, ch in enumerate(chars)}

# Index-to-character mapping
itos = {i: ch for ch, i in stoi.items()}

# Encoding function
def encode(s):
    return [stoi[c] for c in s]

# Decoding function
def decode(indices):
    return ''.join([itos[i] for i in indices])

# Convert the text into integer indices
data = torch.tensor(
    encode(text),
    dtype=torch.long
)

# Length of each input sequence
seq_length = 64


# ============================================================
# Define a simplified Transformer model
# ============================================================
class TinyTransformer(nn.Module):

    def __init__(self, vocab_size, n_embed=64):

        super().__init__()

        # Token embedding layer
        self.token_embed = nn.Embedding(
            vocab_size,
            n_embed
        )

        # Positional embedding layer
        self.pos_embed = nn.Embedding(
            seq_length,
            n_embed
        )

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=n_embed,
            nhead=4,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        # Output layer
        self.fc = nn.Linear(
            n_embed,
            vocab_size
        )

    def forward(self, x):

        # x shape: [batch_size, seq_length]
        B, T = x.shape

        # Create position indices
        pos = torch.arange(
            T,
            device=x.device
        ).unsqueeze(0)

        # Token embeddings + positional embeddings
        x = (
            self.token_embed(x)
            + self.pos_embed(pos)
        )

        # Transformer encoder
        x = self.transformer(x)

        # Output logits
        logits = self.fc(x)

        return logits


# ============================================================
# Initialize model, optimizer, and loss function
# ============================================================
vocab_size = len(chars)

model = TinyTransformer(vocab_size)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

loss_fn = nn.CrossEntropyLoss()

# Text Training Example

In [ ]:
# ============================================================
# Construct a mini-batch
# ============================================================
def get_batch():

    # Randomly select 32 starting positions
    idx = torch.randint(
        0,
        len(data) - seq_length - 1,
        (32,)
    )

    # Input sequences
    X = torch.stack(
        [
            data[i:i + seq_length]
            for i in idx
        ]
    )

    # Target sequences
    y = torch.stack(
        [
            data[i + 1:i + seq_length + 1]
            for i in idx
        ]
    )

    return X, y


# ============================================================
# Main training loop
# ============================================================
for epoch in range(100):

    # Training mode
    model.train()

    # Obtain a mini-batch
    x_batch, y_batch = get_batch()

    # Forward pass
    logits = model(x_batch)

    # Compute cross-entropy loss
    loss = loss_fn(
        logits.view(-1, vocab_size),
        y_batch.view(-1)
    )

    # Clear previous gradients
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Update parameters
    optimizer.step()

    # Print the loss every 10 epochs
    if epoch % 10 == 0:
        print(
            f"Epoch {epoch}, Loss: {loss.item():.4f}"
        )

# Text Generation Testing

In [ ]:
# ============================================================
# Text generation function
# ============================================================
def generate(
    model,
    start='T',
    length=200
):

    # Evaluation mode
    model.eval()

    # Encode the initial prompt
    input_ids = torch.tensor(
        [stoi[c] for c in start],
        dtype=torch.long
    ).unsqueeze(0)

    # Generate characters iteratively
    for _ in range(length):

        # Use the most recent seq_length characters
        x = input_ids[:, -seq_length:]

        # Left-pad if necessary
        pad = seq_length - x.shape[1]

        if pad > 0:

            x = torch.cat(
                [
                    torch.zeros(
                        (1, pad),
                        dtype=torch.long
                    ),
                    x
                ],
                dim=1
            )

        # Forward pass
        logits = model(x)

        # Probability distribution for the next character
        probs = torch.softmax(
            logits[0, -1],
            dim=0
        )

        # Sample the next character
        next_char = torch.multinomial(
            probs,
            num_samples=1
        ).item()

        # Append the new character
        input_ids = torch.cat(
            [
                input_ids,
                torch.tensor([[next_char]])
            ],
            dim=1
        )

    # Decode and return the generated text
    return decode(
        input_ids[0].tolist()
    )


# ============================================================
# Example
# ============================================================
print(
    generate(
        model,
        start="To be, or not to be",
        length=200
    )
)